# Murmur Mamba3 MIMO — 40M smoke on Kaggle T4

This is an experimental, from-scratch run. It preserves the GQA smoke unchanged. Enable GPU and Internet; attach `almirneto/corpus-c4` with **Add Data**. The notebook stops before training if official Mamba3 MIMO cannot compile and run on the current T4.


In [ ]:
# 1. Project environment
from pathlib import Path
import subprocess, sys
repo_dir = Path('/kaggle/working/science')
if not (repo_dir / 'pyproject.toml').exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/orkrs/murmur-science.git', str(repo_dir)])
%cd /kaggle/working/science
%pip install -q --no-deps -e .
sys.path[:0] = ['/kaggle/working/science/src', '/kaggle/working/science']
Path('artifacts').mkdir(exist_ok=True)
print('Environment ready')

In [ ]:
# 2. GPU and official Mamba3 MIMO installation
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable Kaggle GPU: Settings -> Accelerator -> GPU, then restart.')
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))
!nvidia-smi
# The official implementation requires its TileLang MIMO kernel.
%pip install -q tilelang einops
%pip install -q --no-build-isolation git+https://github.com/state-spaces/mamba.git@main
!python scripts/hardware_probe.py --output artifacts/hardware_mamba3_mimo.json

In [ ]:
# 3. Mandatory MIMO hardware gate — do not remove this cell
import json
from mamba_ssm.modules.mamba3 import Mamba3
try:
    gate = Mamba3(d_model=128, d_state=64, headdim=64, is_mimo=True, mimo_rank=2, chunk_size=32).cuda().half().train()
    x = torch.randn(1, 128, 128, device='cuda', dtype=torch.float16, requires_grad=True)
    y = gate(x)
    y.float().square().mean().backward()
    if not torch.isfinite(y).all() or not torch.isfinite(x.grad).all():
        raise RuntimeError('MIMO forward/backward returned non-finite tensors')
    gate_result = {'passed': True, 'gpu': torch.cuda.get_device_name(0), 'output_shape': list(y.shape)}
except Exception as exc:
    gate_result = {'passed': False, 'error': repr(exc)}
Path('artifacts/mamba3_mimo_gate.json').write_text(json.dumps(gate_result, indent=2))
print(gate_result)
if not gate_result['passed']:
    raise RuntimeError('Mamba3 MIMO gate failed. Do not start the smoke; save this log and send it to me.')

In [ ]:
# 4. Verify the experimental configuration
!python scripts/param_count.py --config configs/smoke_mamba3_mimo.toml
from murmur.config import load_run_config
config = load_run_config(Path('configs/smoke_mamba3_mimo.toml'))
assert config.model.mixer == 'mamba3_mimo'
print('MIMO rank is fixed to 2; GQA baseline files were not modified.')

In [ ]:
# 5. Build a 20M-token C4 smoke corpus from Kaggle Add Data
# Expected attached path: /kaggle/input/corpus-c4. The search also handles a renamed Kaggle mount.
import csv, hashlib, json, re
import pandas as pd
raw_candidates = [p for p in Path('/kaggle/input').iterdir() if p.is_dir() and ('c4' in p.name.lower() or 'corpus' in p.name.lower())]
if not raw_candidates:
    raise FileNotFoundError('Attach almirneto/corpus-c4 through Add Data, then re-run this cell.')
raw_dir = raw_candidates[0]
out = Path('artifacts/mimo_smoke_corpus'); out.mkdir(parents=True, exist_ok=True)
budget, accepted, seen, train, val = 20_000_000, 0, set(), [], []
def clean(text):
    return re.sub(r'\s+', ' ', str(text)).strip()
def add(text):
    global accepted
    text = clean(text)
    if len(text) < 80: return False
    digest = hashlib.sha256(text.encode()).hexdigest()
    if digest in seen: return False
    estimate = max(1, len(text.encode()) // 4)
    if accepted + estimate > budget: return False
    seen.add(digest); accepted += estimate
    row = {'text': text, 'source': 'almirneto/corpus-c4', 'text_sha256': digest}
    (val if int(digest[:8], 16) % 100 < 2 else train).append(row)
    return True
for path in sorted(raw_dir.rglob('*')):
    if accepted >= budget: break
    suffix = path.suffix.lower()
    if suffix == '.csv':
        for chunk in pd.read_csv(path, chunksize=10_000):
            col = next((c for c in ('text','content','body') if c in chunk.columns), None)
            if col:
                for text in chunk[col].dropna():
                    add(text)
                    if accepted >= budget: break
    elif suffix == '.parquet':
        frame = pd.read_parquet(path, columns=None)
        col = next((c for c in ('text','content','body') if c in frame.columns), None)
        if col:
            for text in frame[col].dropna():
                add(text)
                if accepted >= budget: break
for name, rows in [('train.jsonl', train), ('val.jsonl', val)]:
    with (out / name).open('w', encoding='utf-8') as f:
        for row in rows: f.write(json.dumps(row, ensure_ascii=False) + '\n')
with (out / 'corpus.txt').open('w', encoding='utf-8') as f:
    for row in train + val: f.write(row['text'] + '\n\n')
if not train or not val: raise RuntimeError('Corpus conversion produced an empty split')
print({'train_docs': len(train), 'val_docs': len(val), 'estimated_tokens': accepted})

In [ ]:
# 6. Train a fresh byte-fallback tokenizer and pack the data
!python scripts/train_tokenizer.py --corpus artifacts/mimo_smoke_corpus/corpus.txt --output artifacts/mimo_smoke_tokenizer.model --vocab-size 32000
!python scripts/prepare_data.py --config configs/smoke_mamba3_mimo.toml --tokenizer artifacts/mimo_smoke_tokenizer.model --train-input artifacts/mimo_smoke_corpus/train.jsonl --val-input artifacts/mimo_smoke_corpus/val.jsonl --output artifacts/mimo_smoke_data
assert list(Path('artifacts/mimo_smoke_data').glob('train_*.bin'))
assert list(Path('artifacts/mimo_smoke_data').glob('val_*.bin'))
print('Packed smoke data ready')

In [ ]:
# 7. Redirect the config to this run's packed data and train from random weights
template = Path('configs/smoke_mamba3_mimo.toml').read_text()
session_config = template.replace('artifacts/data/train.bin', 'artifacts/mimo_smoke_data/train.bin').replace('artifacts/data/val.bin', 'artifacts/mimo_smoke_data/val.bin')
Path('configs/smoke_mamba3_mimo_session.toml').write_text(session_config)
run_dir = Path('artifacts/runs/mamba3_mimo_40m_smoke')
subprocess.run(['python', 'scripts/train.py', '--config', 'configs/smoke_mamba3_mimo_session.toml', '--run-dir', str(run_dir), '--device', 'cuda'], check=True)
assert (run_dir / 'checkpoints' / 'last' / 'COMPLETED').exists()
print('Mamba3 MIMO smoke checkpoint ready')

In [ ]:
# 8. Full-sequence evaluation. Cached generation is intentionally disabled until MIMO cache parity passes.
!python scripts/evaluate.py --config configs/smoke_mamba3_mimo_session.toml --checkpoint artifacts/runs/mamba3_mimo_40m_smoke/checkpoints/last --output artifacts/eval_mamba3_mimo_40m.json --device cuda
print(Path('artifacts/eval_mamba3_mimo_40m.json').read_text())
print('Upload artifacts/mamba3_mimo_gate.json, training output and eval JSON for analysis.')